# 2026 Global and regional hackathon

* For the 2026 hackathon: https://digital-earths-uk-hackathon.github.io/
* This notebook will get you started with the global, N2560 RAL3.3 (tuned) DYAMOND3-class model.
* It can be run locally or on JASMIN by loading the 'UK' catalog or from anywhere by opening the 'online' catalog

## Open dataset from catalog

In [ ]:
import cartopy.crs as ccrs
import intake
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import easygems.healpix as egh

from utils import hp_mods, plot_all_fields

In [ ]:
# Filter out annoying warning.
import warnings
warnings.filterwarnings("ignore", message=".*The return type of `Dataset.dims` will be changed.*", category=FutureWarning)

In [ ]:
# Open catalog and print hk26 available models.
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['UK']
# Use online if not on JASMIN.
# cat = intake.open_catalog('https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml')['online']
print('\n'.join([k for k in cat if k.endswith('_hk26')]))

In [ ]:
# Load specific model.
sim = 'um_glm_n2560_RAL3p3_tuned_hk26'
sim_cat = cat[sim]

In [ ]:
# Open a 1h (2D) and 3h (3D) dataset.
ds1h = sim_cat(zoom=8, time='PT1H').to_dask().pipe(hp_mods)
ds3h = sim_cat(zoom=8, time='PT3H').to_dask().pipe(hp_mods)

## Have a look at data

In [ ]:
# Look at some data.
ds1h

In [ ]:
# The variable_ids are somewhat cryptic - let's see what they mean...
for var_id, da in ds1h.data_vars.items():
    if var_id == 'weights':
        continue
    print(f'{var_id:<6}: {da.attrs["standard_name"]} ({da.attrs["units"]})')
print()
for var_id, da in ds3h.data_vars.items():
    if var_id == 'weights':
        continue
    print(f'{var_id:<6}: {da.attrs["standard_name"]} ({da.attrs["units"]})')


## Display data

In [ ]:
# How can I plot the data?
# What is 'tas'?
egh.healpix_show(ds1h.tas.isel(time=13))

In [ ]:
# Let's see all the hourly fields at the second timestamp (00:00 has some variables without data).
# .compute() loads all of the data from the object store into RAM.
ds1h_plot = ds1h.sel(time=pd.Timestamp('2020-01-20 01:00')).compute()

In [ ]:
# Plot all fields!
plot_all_fields(ds1h_plot)

In [ ]:
# 3-hourly data (3D data) at 900 hPa.
ds3h_plot = ds3h.sel(time=pd.Timestamp('2020-01-20 03:00')).sel(pressure=900).compute()

In [ ]:
plot_all_fields(ds3h_plot)

In [ ]:
# Check that there's more cloud liquid water in the lower troposphere than stratosphere.
ds3h.isel(time=1).clw.mean(dim='cell').plot(y='pressure', yincrease=False)

## Subset to particular region

* Note, this is harder if you want to cross the 0deg longitude line - how would you do this?

In [ ]:
# How do I subset to a particular region? Can I control the plotting?
aus_mask = (ds1h.lon > 110) & (ds1h.lon < 160) & (ds1h.lat > -45) & (ds1h.lat < -10)
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, layout='constrained')
ax.set_global()
ax.coastlines()
egh.healpix_show(ds1h.tas.isel(time=13).where(aus_mask), ax=ax)

## Look at timeseries data

In [ ]:
# We can load the data at different zooms. What are the time chunks for zoom 0? Why is this useful?
ds1h_z0 = sim_cat(zoom=0, time='PT1H').to_dask().pipe(hp_mods)
ds1h_z0

In [ ]:
ds1h_z0.pr.mean(dim='cell').plot()

In [ ]:
# A bit more realistic. Timeseries of precip over Australia at zoom=4. You could try doing at zoom=10 but you might be here a while...
# For reference: z=4 took 2.15s, z=10 took 11m54.94s.
ds1h_z4 = sim_cat(zoom=4, time='PT1H').to_dask().pipe(hp_mods)

aus_mask = (ds1h_z4.lon > 110) & (ds1h_z4.lon < 160) & (ds1h_z4.lat > -45) & (ds1h_z4.lat < -10)
aus_pr = ds1h_z4.sel(cell=aus_mask).pr
# Show the region (at zoom=4)
egh.healpix_show(aus_pr.mean(dim='time'))
# Show the timeseries of precip.
plt.figure()
aus_pr.sum(dim='cell').plot()

## Zonal mean

In [ ]:
def plot_zonal_mean(da, ax=None, bins=np.linspace(-90, 90, 181), **kwargs):
    if ax is None:
        fig, ax = plt.subplots(layout='constrained')

    bin_mids = (bins[:-1] + bins[1:]) / 2
    zoom = da.crs.attrs['refinement_level']
    zonal_mean_da = da.mean(dim='time').groupby_bins('lat', bins).mean()
    plt.plot(bin_mids, zonal_mean_da, label=f'z{zoom}', **kwargs)

    plt.xlim((-90, 90))
    plt.xlabel('lat')
    plt.ylabel(f'{da.long_name} ({da.attrs.get("units", "-")})')

In [ ]:
ds1h_z6 = sim_cat(zoom=6, time='PT1H').to_dask().pipe(hp_mods)
ds1h_z5 = sim_cat(zoom=5, time='PT1H').to_dask().pipe(hp_mods)
ds1h_z4 = sim_cat(zoom=4, time='PT1H').to_dask().pipe(hp_mods)
fig, ax = plt.subplots(layout='constrained')

plot_zonal_mean(ds1h.pr.isel(time=slice(1, 24 * 30)), ax=ax, bins=np.linspace(-90, 90, 181))
plot_zonal_mean(ds1h_z6.pr.isel(time=slice(1, 24 * 30)), ax=ax, bins=np.linspace(-90, 90, 181))
plot_zonal_mean(ds1h_z5.pr.isel(time=slice(1, 24 * 30)), ax=ax, bins=np.linspace(-90, 90, 181))
plot_zonal_mean(ds1h_z4.pr.isel(time=slice(1, 24 * 30)), ax=ax, bins=np.linspace(-90, 90, 46))

ax.legend()